# [Chapter 3 — Force of Infection (Susceptible Viewpoint), §3.1-3.3] The Force of Infection $\lambda = c_S \beta P_I$

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 3 — Force of Infection (Susceptible Viewpoint)
**Considerations developed:** 4 (force of vs from), 5 (transmission probability)
**Estimated runtime:** ≤ 30 seconds

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Develops the force of infection $\lambda = c_S \beta P_I$ from the susceptible viewpoint, demonstrates the linear-vs-exponential approximation, and prepares the contrast with Chapter 2B's infected viewpoint.

## What you should already know
Chapter 2 notebook on the infection equation.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import set_seed_chapter_02, book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_07

set_seed_chapter_02()
book_style()


## The susceptible viewpoint

A susceptible individual encounters infectious individuals at a rate $c_S$ contacts per day, with each contact carrying probability $\beta$ of transmission, and the fraction of contacts that are with infectious individuals being $P_I = I/N$. Therefore:

$$\lambda = c_S \beta P_I = c_S \beta \frac{I}{N}$$

is the per-susceptible rate of becoming infected, and the total incidence is $J = \lambda S$.

This is the *conventional* form found in most introductory textbooks. It frames infection as something that *happens to* susceptibles.

In [ ]:
# Plot lambda over time during a typical outbreak
params = baseline_chapter_07()
result = integrate_sir_i(params)

t = result['t']
S = result['S']
I = result['I']
N = params['N']
c_S = params['c_S']
beta = params['beta']

# Force of infection over time
lambda_t = c_S * beta * (I / N)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(t, I, color=BOOK_COLORS['infectious'], lw=1.5, label='I(t) (infectious fraction)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('I/N')
ax.set_title('Infectious fraction over time')
ax.legend()

ax = axes[1]
ax.plot(t, lambda_t, color=BOOK_COLORS['lambda_hat'], lw=1.5, label='$\\lambda(t)$ (force of infection)')
ax.set_xlabel('Time (days)')
ax.set_ylabel('$\\lambda$ (per day)')
ax.set_title('Force of infection: $\\lambda = c_S \\beta I / N$')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Peak lambda: {lambda_t.max():.4f} per day at t = {t[lambda_t.argmax()]:.1f} days')
print(f'lambda is proportional to I(t); both peak at the same time.')


## The linear-vs-exponential approximation

Early in an outbreak when $S \approx N$, the per-susceptible probability of escaping infection over time $\Delta t$ can be approximated two ways:

- **Linear**: $1 - \lambda \Delta t$ (valid when $\lambda \Delta t \ll 1$)
- **Exponential**: $e^{-\lambda \Delta t}$ (the exact form for a memoryless process)

The book uses the linear form throughout for tractability. This approximation breaks down when $\lambda \Delta t$ approaches 1 — i.e., during the peak.

In [ ]:
# Compare linear vs exponential approximation across lambda * dt range
ldt = np.linspace(0, 1.5, 100)
linear = 1 - ldt
exponential = np.exp(-ldt)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(ldt, linear, color=BOOK_COLORS['susceptible'], lw=1.5, label='Linear: $1 - \\lambda \\Delta t$')
ax.plot(ldt, exponential, color=BOOK_COLORS['infectious'], lw=1.5, label='Exact: $e^{-\\lambda \\Delta t}$')
ax.fill_between(ldt, linear, exponential, alpha=0.15, color=BOOK_COLORS['highlight'])
ax.axhline(0, color='k', lw=0.5, alpha=0.3)
ax.axvline(0.1, color=BOOK_COLORS['negative'], ls=':', lw=1, alpha=0.7, label='$\\lambda \\Delta t = 0.1$ (linear OK)')
ax.set_xlabel('$\\lambda \\Delta t$')
ax.set_ylabel('Probability of escape')
ax.set_title('Linear vs exact escape probability')
ax.legend()
ax.set_xlim(0, 1.5)
plt.tight_layout()
plt.show()

# Quantify the error at typical operating points
print(f'At lambda*dt = 0.1: linear = {1-0.1:.4f}, exact = {np.exp(-0.1):.4f}, error = {abs(1-0.1 - np.exp(-0.1))*100:.2f}%')
print(f'At lambda*dt = 0.5: linear = {1-0.5:.4f}, exact = {np.exp(-0.5):.4f}, error = {abs(1-0.5 - np.exp(-0.5))*100:.2f}%')
print(f'At lambda*dt = 1.0: linear = {max(1-1.0, 0):.4f}, exact = {np.exp(-1.0):.4f}, error large.')


## What's next

The susceptible viewpoint is conventional but has a subtle limitation: the data don't usually observe $S$ directly — they observe new infections (which come from $I$). The next notebook (**Chapter 2B**) introduces the **infected viewpoint** $\alpha = c_I \beta P_S$ that the book uses as its primary form.